![NVIDIA Logo](images/nvidia.png)

# Exercise: Filter Messages

For this exercise, you will be creating a custom stage that utilizes `Broadcast` that can filter the data in an incoming message based on a filter condition. The custom stage will then send the filtered data to one output port, while passing through the entire message to a second output port.

---

## Filter Function

`filter_message_meta` can be used to assist your work. It expects a `MessageMeta` and a `filter_condition` function, returning a `MessageMeta` containing only the filtered data.

In [1]:
import cudf
from morpheus.messages import MessageMeta

def filter_message_meta(message: MessageMeta, filter_condition) -> MessageMeta:
    """
    Filters a MessageMeta's dataframe using a provided filter condition.

    Args:
        message (MessageMeta): The input message containing a cudf DataFrame.
        filter_condition (callable): A function that takes a cudf DataFrame and returns a filtered DataFrame.

    Returns:
        MessageMeta: A new MessageMeta containing only the filtered data.
    """
    with message.mutable_dataframe() as df:
        filtered_df = filter_condition(df)

    return MessageMeta(filtered_df)

We'll try it out with our simple user authentication logs data.

In [2]:
input_file = 'data/simple_user_log.jsonlines'

In [3]:
df = cudf.read_json(input_file, lines=True)
mm = MessageMeta(df)

In [4]:
df

,timestamp,user,ip_address,request_time,status,error_message
0,2025-02-01T10:15:30Z,user123,192.168.1.10,200.45,success,<NA>
1,2025-02-01T10:17:00Z,user123,192.168.1.20,150.55,failure,Invalid credentials
2,2025-02-01T10:18:10Z,user456,10.0.0.5,180.60,success,<NA>
3,2025-02-01T10:19:25Z,user789,192.168.1.30,215.25,failure,Timeout
4,2025-02-01T10:20:00Z,user456,10.0.0.6,120.10,success,<NA>
5,2025-02-01T10:22:30Z,user123,192.168.1.40,175.35,failure,Access denied
6,2025-02-01T10:23:45Z,user321,192.168.1.50,205.50,success,<NA>
7,2025-02-01T10:25:05Z,user864,192.168.1.60,190.15,failure,Invalid session
8,2025-02-01T10:26:20Z,user123,192.168.1.70,210.80,success,<NA>
9,2025-02-01T10:27:40Z,user456,10.0.0.7,160.95,failure,Account locked


Here we create two filter condition functions, which need to expect and return dataframes.

In [5]:
def filter_success(df):
    return df[df["status"] == "success"]

def filter_failure(df):
    return df[df["status"] == "failure"]

Now we can try our `filter_message_meta` function out with our dataframe, and each of the filter functions.

In [6]:
filter_message_meta(mm, filter_success).get_data()

,timestamp,user,ip_address,request_time,status,error_message
0,2025-02-01T10:15:30Z,user123,192.168.1.10,200.45,success,<NA>
2,2025-02-01T10:18:10Z,user456,10.0.0.5,180.60,success,<NA>
4,2025-02-01T10:20:00Z,user456,10.0.0.6,120.10,success,<NA>
6,2025-02-01T10:23:45Z,user321,192.168.1.50,205.50,success,<NA>
8,2025-02-01T10:26:20Z,user123,192.168.1.70,210.80,success,<NA>


In [7]:
filter_message_meta(mm, filter_failure).get_data()

,timestamp,user,ip_address,request_time,status,error_message
1,2025-02-01T10:17:00Z,user123,192.168.1.20,150.55,failure,Invalid credentials
3,2025-02-01T10:19:25Z,user789,192.168.1.30,215.25,failure,Timeout
5,2025-02-01T10:22:30Z,user123,192.168.1.40,175.35,failure,Access denied
7,2025-02-01T10:25:05Z,user864,192.168.1.60,190.15,failure,Invalid session
9,2025-02-01T10:27:40Z,user456,10.0.0.7,160.95,failure,Account locked


---

## Imports

You will likely need to use the following imports in your work.

In [8]:
import logging
import typing

from IPython.display import Image
import cudf

from morpheus.config import Config

from morpheus.pipeline import Pipeline
from morpheus.pipeline.execution_mode_mixins import GpuAndCpuMixin
from morpheus.pipeline.pass_thru_type_mixin import PassThruTypeMixin
from morpheus.pipeline.single_port_stage import SinglePortStage

from morpheus.pipeline.stage import Stage
from morpheus.pipeline.stage_schema import StageSchema

from morpheus.stages.input.file_source_stage import FileSourceStage
from morpheus.stages.output.in_memory_sink_stage import InMemorySinkStage
from morpheus.stages.output.write_to_file_stage import WriteToFileStage

from morpheus.messages import MessageMeta

from morpheus.utils.logger import configure_logging, reset_logging

import mrc
import mrc.core.operators as ops
from mrc.core.node import Broadcast

---

## Exercise Objectives

Create a custom stage that utilizes `Broadcast` that can filter the data in an incoming message based on a filter condition. The custom stage will then send the filtered data to one output port, while passing through the entire message to a second output port.

You will likely wish to use `filter_message_meta` to support your work.

We recommend parameterizing the filter condition function so the custom stage can be reused to filter on a variety of conditions. For example, you should be able to do both of the following:

```python
filter_succsess_stage = pipeline.add_stage(YourCustomStage(config, filter_condition=filter_success))
filter_failure_stage = pipeline.add_stage(YourCustomStage(config, filter_condition=filter_failure))
```

## Your Work Here

Build and run your pipeline in the space provided below. By all means feel free to create additional code cells for your work, which you can do by clicking the `+` button in the Jupyter menu bar at the top of this notebook.

If you get stuck, a solution is provided below, which you view by expanding the *Solution* section below.

In [ ]:
class FilteredBroadcastStage(GpuAndCpuMixin, Stage):
    def __init__(self, c: Config, filter_condition):
        super().__init__(c)

        self._create_ports(1, 2)  # One input port, two output ports
        self._filter_condition = filter_condition  # Store the filter function

    @property
    def name(self) -> str:
        return "filtered-broadcast"

    def supports_cpp_node(self):
        return False

    def accepted_types(self) -> tuple:
        return (MessageMeta, )

    def compute_schema(self, schema: StageSchema):
        for port_schema in schema.output_schemas:
            port_schema.set_type(MessageMeta)

    def filter_data(self, message: MessageMeta) -> MessageMeta:

        with message.mutable_dataframe() as df:
            filtered_df = self._filter_condition(df)

        return MessageMeta(filtered_df)

    def _build(self, builder: mrc.Builder, input_nodes: list[mrc.SegmentObject]) -> list[mrc.SegmentObject]:

        # Create a broadcast node
        broadcast = Broadcast(builder, "broadcast")
        builder.make_edge(input_nodes[0], broadcast)

        # Create outgoing nodes
        filter_node = builder.make_node("filtered_output", ops.map(self.filter_data))
        passthrough_node = builder.make_node("passthrough_output", ops.map(lambda msg: msg))

        # Connect broadcast to both nodes
        builder.make_edge(broadcast, filter_node)
        builder.make_edge(broadcast, passthrough_node)

        return [filter_node, passthrough_node]

In [ ]:
config = Config()
pipeline = Pipeline(config)

source = pipeline.add_stage(FileSourceStage(config, filename=input_file, iterative=False))
filter_stage = pipeline.add_stage(FilteredBroadcastStage(config, filter_condition=filter_success))
pipeline.add_edge(source, filter_stage)

in_mem_sink_1 = pipeline.add_stage(InMemorySinkStage(config))
in_mem_sink_2 = pipeline.add_stage(InMemorySinkStage(config))

pipeline.add_edge(filter_stage.output_ports[0], in_mem_sink_1)
pipeline.add_edge(filter_stage.output_ports[1], in_mem_sink_2)

In [ ]:
pipeline.build()

In [ ]:
viz_file = './pipeline_visualizations/filter.png'
pipeline.visualize(viz_file)

In [ ]:
Image(filename=viz_file)

In [ ]:
reset_logging()
configure_logging(log_level=logging.DEBUG)

---

## Solution

In [ ]:
class FilteredBroadcastStage(GpuAndCpuMixin, Stage):
    def __init__(self, c: Config, filter_condition):
        super().__init__(c)

        self._create_ports(1, 2)  # One input port, two output ports
        self._filter_condition = filter_condition  # Store the filter function

    @property
    def name(self) -> str:
        return "filtered-broadcast"

    def supports_cpp_node(self):
        return False

    def accepted_types(self) -> tuple:
        return (MessageMeta, )

    def compute_schema(self, schema: StageSchema):
        for port_schema in schema.output_schemas:
            port_schema.set_type(MessageMeta)

    def filter_data(self, message: MessageMeta) -> MessageMeta:

        with message.mutable_dataframe() as df:
            filtered_df = self._filter_condition(df)

        return MessageMeta(filtered_df)

    def _build(self, builder: mrc.Builder, input_nodes: list[mrc.SegmentObject]) -> list[mrc.SegmentObject]:

        # Create a broadcast node
        broadcast = Broadcast(builder, "broadcast")
        builder.make_edge(input_nodes[0], broadcast)

        # Create outgoing nodes
        filter_node = builder.make_node("filtered_output", ops.map(self.filter_data))
        passthrough_node = builder.make_node("passthrough_output", ops.map(lambda msg: msg))

        # Connect broadcast to both nodes
        builder.make_edge(broadcast, filter_node)
        builder.make_edge(broadcast, passthrough_node)

        return [filter_node, passthrough_node]

In [ ]:
config = Config()
pipeline = Pipeline(config)

source = pipeline.add_stage(FileSourceStage(config, filename=input_file, iterative=False))
filter_stage = pipeline.add_stage(FilteredBroadcastStage(config, filter_condition=filter_success))
pipeline.add_edge(source, filter_stage)

in_mem_sink_1 = pipeline.add_stage(InMemorySinkStage(config))
in_mem_sink_2 = pipeline.add_stage(InMemorySinkStage(config))

pipeline.add_edge(filter_stage.output_ports[0], in_mem_sink_1)
pipeline.add_edge(filter_stage.output_ports[1], in_mem_sink_2)

In [ ]:
pipeline.build()

In [ ]:
viz_file = './pipeline_visualizations/filter.png'
pipeline.visualize(viz_file)

In [ ]:
Image(filename=viz_file)

In [ ]:
reset_logging()
configure_logging(log_level=logging.DEBUG)

In [ ]:
await pipeline.run_async()

In [ ]:
filtered_messages = in_mem_sink_1.get_messages()
filtered_messages[0].df

In [ ]:
all_messages = in_mem_sink_2.get_messages()
all_messages[0].df